# 🩺 HealthPilot — OpenBioLLM **Base-Model Capability Test** (annotated)

**Purpose:** evaluate the triage quality of the **base** `aaditya/Llama3-OpenBioLLM-8B`
model *as-is* (no fine-tuning, no adapter) using only the HealthPilot **system prompt**.
This is **not** a smoke test and **not** the fine-tuned model — it tests what the
off-the-shelf medical model can already do.

**Why this notebook is important — the key project finding:**
In side-by-side testing on the same emergency case, the **base OpenBioLLM here triaged
CORRECTLY** ("this is a heart attack — go to the ER / call emergency services"), while
the variant **fine-tuned on the raw ChatDoctor dataset under-triaged** the same case
(suggested a stress test / cardiologist referral). In other words, fine-tuning on the
noisy ChatDoctor data **degraded** the model's emergency judgment that pretraining had
already encoded.

**What this implies for HealthPilot:**
- The base model's medical instincts are a strong starting point on their own.
- Fine-tuning on unfiltered ChatDoctor is actively harmful for triage — any future
  fine-tune needs a cleaned dataset (and a gentler touch) so it *preserves* these
  instincts rather than overwriting them.
- It's worth seriously comparing this base model (and MedGemma) against the fine-tune
  before committing to a training-heavy path.

> ⚠️ **Safety:** research/prototype check, not validated clinical software. A correct
> answer on one case is **not** validated triage. The urgent / 999 / booking decision
> must come from a deterministic **rule layer** reading the symptoms/upstream-node
> outputs (biased toward over-triage) — never from the model alone.

**Prerequisites:**
- GPU runtime (the code calls `.to("cuda")`)
- `user_secrets` available (Kaggle) with an `HF_TOKEN`, for the model download

---
## 1 · System prompt
The persona/behavior used to steer the base model at inference. (It's the same prompt
used in fine-tuning, but here it's doing all the work on its own.)

In [ ]:
# ── HealthPilot system prompt ───────────────────────────────────────────────
# The persona/behavior string. In THIS notebook it steers the BASE OpenBioLLM at
# inference (no fine-tuning involved) — it defines the assistant's role (clinical
# SUPPORT, not a diagnosing physician), the required answer structure, and the
# severity tiers (mild / moderate / red-flag).

SYSTEM_INSTRUCTION = """You are 'HealthPilot', a clinical support assistant that helps patients \
understand their symptoms and decide what to do next. You are not a physician and you do not give a definitive \
diagnosis; you provide careful, well-reasoned guidance and a clear recommendation for next steps.

You are given a patient's described symptoms and relevant medical history. For each case you must:
1. Briefly restate your understanding of the patient's situation in plain, reassuring language.
2. Explain the most relevant considerations for their symptoms, without overstating certainty.
3. Recommend clear next steps, calibrated to severity:
   - Mild / self-limiting: sensible self-care, plus the specific warning signs that should prompt escalation.
   - Moderate: advise being seen by an appropriate clinician soon.
   - Severe or red-flag symptoms (e.g. chest pain, trouble breathing, stroke signs, severe bleeding, \
fainting, signs of sepsis): state clearly that this needs immediate medical attention and recommend \
urgent/emergency care.
4. When uncertain, err on the side of caution and recommend professional evaluation.

Always respond in an organized, professional, and empathetic tone. Use a clear structure. Never dismiss \
potentially serious symptoms. Always close by reminding the patient that this guidance does not replace \
assessment by a qualified healthcare professional."""

---
## 2 · Load the **base** model and generate
Import Unsloth, download and load the **base** OpenBioLLM (no adapter), format the
prompt with the Llama-3.1 chat template, and run one emergency test case. Read the
output critically — this is the cell that revealed the base model triages well.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
# FastLanguageModel  → Unsloth's optimized loader / inference path
# get_chat_template  → attaches the Llama-3.1 chat template (OpenBioLLM's tokenizer
#                      ships without one); REQUIRED before apply_chat_template
# snapshot_download  → pulls the base model repo to a local path (avoids the
#                      safetensors / 'discussions disabled (403)' load quirk)
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from huggingface_hub import snapshot_download

In [ ]:
# ── Download the BASE model ─────────────────────────────────────────────────
# Pulls aaditya/Llama3-OpenBioLLM-8B (the BASE medical model — NOT your fine-tuned
# adapter) to a local path. We test the base model's own capability here.
# Requires user_secrets with HF_TOKEN (Kaggle).
path = snapshot_download(
    repo_id="aaditya/Llama3-OpenBioLLM-8B",
    token=user_secrets.get_secret("HF_TOKEN"),
)

In [ ]:
# ── Load the BASE model + run one test case ─────────────────────────────────
# IMPORTANT: this loads the BASE OpenBioLLM only (the `path` from snapshot_download
# above) — there is NO adapter / NO fine-tuning applied here. This cell measures the
# off-the-shelf model's triage ability, steered purely by the system prompt.
#
# Steps:
#   1) Load the base model in 4-bit + inference mode (for_inference = Unsloth's fast
#      generation path; disables training-only features).
#   2) Attach the Llama-3.1 chat template (must match the formatting OpenBioLLM expects).
#   3) Build a [system, user] message list and render WITH add_generation_prompt=True
#      (inference: we want the model to CONTINUE after the assistant header).
#   4) Generate and decode.
#
# THINGS TO FIX / KNOW:
#   • eos_token_id=[..., 128009]: 128009 is the Llama-3 <|eot_id|> stop token —
#     correct for this Llama-family model. (It would be WRONG for a Gemma model like
#     MedGemma, which uses <end_of_turn>.)
#   • max_new_tokens=400 can cut off long answers mid-sentence — raise to ~1024.
#   • Add repetition_penalty=1.1 if output ever degenerates into repeated tokens.
#
# ✅ RESULT WE OBSERVED (the key finding):
#   On this textbook HEART-ATTACK case, the BASE model answered CORRECTLY —
#   "this sounds like a heart attack, go to the emergency room / call emergency
#   services." By contrast, the ChatDoctor-FINE-TUNED variant under-triaged the SAME
#   case (stress test / cardiologist). => Fine-tuning on raw ChatDoctor DEGRADED the
#   triage the base model already had. Prefer the base model (or a CLEAN fine-tune)
#   for this task.
#
# ⚠️ CLINICAL SAFETY: a correct answer here is NOT validated triage. The urgent/999/
#   booking escalation must come from a deterministic RULE LAYER reading the symptoms/
#   upstream nodes, biased toward over-triage — never the model alone.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=path, max_seq_length=1024, load_in_4bit=True)
FastLanguageModel.for_inference(model)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",   # OpenBioLLM is Llama-3.1-8B based
)

messages = [
    {"role": "system", "content": SYSTEM_INSTRUCTION},
    {"role": "user", "content": "I'm a 45-year-old man with crushing chest pain spreading to my left arm and I feel sweaty and short of breath."},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=400,
                     eos_token_id=[tokenizer.eos_token_id, 128009])  # 128009 = <|eot_id|>
print(tokenizer.decode(out[0], skip_special_tokens=True))